# Capstone build --- Chapter 10: Trajectory Evaluation and Metrics

The workflow of Chapter~8 produces a sequence of steps; Chapter~10 reads that sequence as an object and scores it. A `Trajectory` is the task together with the recorded steps, and it is inspected at several levels: the decision the agent reached, the structure of how it got there, and a summary that rolls the structural metrics together. This is the first build notebook that runs the whole agent, so it loads the real models and the GMS store.

## Build the harness and run one case

`build_complaint_harness` returns the assembled governance harness and its registry; Chapter~12 dissects what it wires. Running it on a routine case returns a `Trajectory` whose records are the `(state, action, observation)` transitions of the loop.

In [ ]:
import json
from pathlib import Path
from forgeloop.agents.capstone import build_complaint_harness
from forgeloop.agents.core import Budget, BudgetTracker, TaskSpec

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
policies_dir = root / 'data' / 'policies'
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())

harness, registry = build_complaint_harness(policies_dir=policies_dir)
case = next(c for c in cases if c['id'] == 'case-002')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
print('message:', case['message'])
print('steps  :', len(traj.records), '| final status:', traj.final_state.status)

## The decision level

The coarsest reading asks what the agent decided and whether that matches what the case expected. For a routine complaint the expected outcome is a drafted response rather than an escalation.

In [ ]:
out = traj.final_state.final_output or {}
print('classification    :', out.get('classification'))
print('recommended action:', out.get('recommended_action'))
print('expected escalate :', case.get('expected_escalation'))

## The structural level

The `metrics` module reads the trajectory's shape: how many steps it took, how many tool calls it made, whether any tool failed, whether it finished cleanly or escalated. `summarize` rolls these into one record, which is what a suite aggregates over many cases in Chapter~17.

In [ ]:
from forgeloop.agents.evaluation import summarize
from forgeloop.agents.evaluation.metrics import (
    step_count, tool_call_count, tool_failure_count, escalated, finished_cleanly,
)
print('steps         :', step_count(traj))
print('tool calls    :', tool_call_count(traj))
print('tool failures :', tool_failure_count(traj))
print('escalated     :', escalated(traj))
print('finished clean:', finished_cleanly(traj))
print('summary       :', summarize(traj))

## The per-step transitions

The trajectory is inspectable step by step, which is what makes a decision auditable: each record names the action proposed and whether its observation reported success. Reading the records back is how a reviewer sees exactly what the agent did.

In [ ]:
for rec in traj.records:
    kind = rec.action.kind
    tool = getattr(rec.action, 'tool_name', '')
    ok = rec.observation.get('success') if rec.observation else ''
    print(f'step {rec.step}: {kind:10s} {tool:18s} success={ok}')

Reading a trajectory at these levels is what separates evaluation from a demo: the decision says whether the outcome was right, the structure says how the agent got there, and the records make each step auditable. Chapter~11 designs the suite of cases these metrics are aggregated over, and Chapter~17 runs it against the finished agent.